# L16 · Orchestrator-workers: dividir un ticket y delegar

*Advanced Topics in Analytics II - Generative AI* · Instructor: Sergio A. Mora Pardo

En [L15](L15-Paralelization.ipynb) los revisores en paralelo eran **fijos**: siempre tres, escritos en el código. A veces no sabemos de antemano cuántas tareas hay. Un cliente de **ACME** puede escribir una pregunta o cinco, sobre políticas distintas. En el patrón **orchestrator-workers**, un modelo (el orquestador) **planea** las tareas, el grafo lanza **un trabajador por tarea** con `Send` y un sintetizador junta los resultados.

```mermaid
flowchart LR
    S([START]) --> O["orchestrator<br/>Claude · plan de tareas"]
    O -->|"Send × N"| W1["worker · tarea 1"]
    O -->|"Send × N"| W2["worker · tarea 2"]
    O -->|"Send × N"| W3["worker · tarea …"]
    W1 & W2 & W3 --> Y["synthesize<br/>Claude · una sola respuesta"]
    Y --> E([END])
```

## Objetivos

1. Lanzar un número variable de nodos con `Send`.
2. Darle a cada trabajador su propia entrada, distinta del estado principal.
3. Juntar los resultados con un reducer.
4. Escribir el trabajador como un **subgrafo** reutilizable.
5. Comparar este patrón con la paralelización fija de L15.

El recorrido realiza **unas 5 llamadas a Claude Haiku** en la sección 3: 1 del plan, 1 por tarea (el plan decide cuántas) y 1 de síntesis. Las secciones 1 y 2 no llaman al modelo.

## 0. Preparación

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_anthropic import ChatAnthropic

# find_dotenv sube por las carpetas hasta encontrar el .env de la raíz
load_dotenv(find_dotenv())

model = ChatAnthropic(
    model="claude-haiku-5-5",
    max_tokens=500,
    thinking={"type": "disabled"},
    timeout=60,
    max_retries=2,
)

In [ ]:
from pathlib import Path

from pypdf import PdfReader


def policy_text(name):
    """El texto completo de una política: cada una tiene dos páginas y cabe entera en el prompt."""
    path = Path("../datasets") / name
    return "\n\n".join(f"[{name}, p. {n}]\n{page.extract_text()}" for n, page in enumerate(PdfReader(path).pages, 1))

In [ ]:
from IPython.display import Markdown, display


def show(graph):
    """Dibuja el grafo como Mermaid: JupyterLab lo renderiza sin conexión a internet."""
    display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

## 1. `Send`: un nodo por elemento

Sin modelo. Una arista condicional puede devolver una **lista de `Send`**. Cada `Send("worker", entrada)` lanza el nodo `worker` con **su propia entrada**, no con el estado completo, y todos corren en el mismo paso. El reducer de `answers` junta lo que devuelve cada uno.

Aquí el «plan» es una lista escrita a mano: en la sección 3 la escribirá el modelo.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


class TicketState(TypedDict, total=False):
    message: str
    tasks: list[dict]
    answers: Annotated[list[str], operator.add]
    reply: str


def fixed_plan(state: TicketState) -> dict:
    return {"tasks": [{"question": "¿Puedo pagar por PSE?"}, {"question": "¿Cuántas veces puedo reestructurar?"}]}


def assign(state: TicketState) -> list[Send]:
    return [Send("echo_worker", task) for task in state["tasks"]]


def echo_worker(task: dict) -> dict:
    return {"answers": [f"(sin modelo) respondería: {task['question']}"]}


builder = StateGraph(TicketState)
builder.add_node("plan", fixed_plan)
builder.add_node("echo_worker", echo_worker)
builder.add_edge(START, "plan")
builder.add_conditional_edges("plan", assign, ["echo_worker"])
builder.add_edge("echo_worker", END)
demo = builder.compile()
show(demo)

demo.invoke({"message": "dos preguntas"})["answers"]

`add_conditional_edges` recibe la lista de destinos posibles (`["echo_worker"]`) porque LangGraph no puede adivinar a dónde irán los `Send` para dibujar el grafo. Con un plan de cinco tareas habría cinco trabajadores; con una, uno solo.

## 2. El trabajador como subgrafo

Sin modelo todavía en la definición. Cada trabajador hace dos pasos: cargar la política de su tarea y responder con ella. Lo escribimos como un **grafo propio**, con su propio estado, y lo usamos como **un nodo** del grafo principal. Así el trabajador se puede probar solo y reutilizar en otro flujo.

```mermaid
flowchart LR
    subgraph worker["subgrafo worker"]
        L["load_policy"] --> A["answer<br/>Claude"]
    end
```

In [ ]:
from typing import Literal

from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field

Policy = Literal["acme_credit_policy.pdf", "acme_disbursement_policy.pdf", "acme_fees_guide.pdf", "acme_collections_policy.pdf"]


class WorkerState(TypedDict, total=False):
    question: str
    policy: str
    context: str
    answers: Annotated[list[str], operator.add]


def load_policy(state: WorkerState) -> dict:
    return {"context": policy_text(state["policy"])}


def answer(state: WorkerState) -> dict:
    prompt = [
        SystemMessage("Responde en español, en una o dos frases, solo con la política, y cita con [archivo, p. N]."),
        HumanMessage(f"Política:\n{state['context']}\n\nPregunta: {state['question']}"),
    ]
    return {"answers": [f"**{state['question']}** {model.invoke(prompt).text}"]}


worker_builder = StateGraph(WorkerState)
worker_builder.add_sequence([load_policy, answer])
worker_builder.add_edge(START, "load_policy")
worker = worker_builder.compile()
show(worker)

## 3. El orquestador, los trabajadores y la síntesis

El orquestador lee el ticket y devuelve un **plan**: una lista de tareas, cada una con su pregunta y la política que la responde (salida estructurada, L11). `assign_workers` lanza un `Send` por tarea hacia el subgrafo; si el plan llega vacío, va directo a `synthesize`. `synthesize` espera a todos y escribe **una** respuesta para el cliente.

In [ ]:
class Task(BaseModel):
    """Una pregunta del ticket y la política que la responde."""

    question: str = Field(description="Una sola pregunta del cliente, completa y en una frase.")
    policy: Policy = Field(description="crédito, desembolsos, costos y pagos, o cobranza.")


class Plan(BaseModel):
    """Las preguntas distintas que trae el ticket."""

    tasks: list[Task] = Field(description="Una tarea por cada pregunta distinta del cliente.")


def orchestrator(state: TicketState) -> dict:
    plan = model.with_structured_output(Plan).invoke([HumanMessage(state["message"])])
    return {"tasks": [task.model_dump() for task in plan.tasks]}


def assign_workers(state: TicketState) -> list[Send] | str:
    # Un plan vacío no lanza trabajadores: sin este caso, el grafo terminaría sin respuesta.
    return [Send("worker", task) for task in state["tasks"]] or "synthesize"


def synthesize(state: TicketState) -> dict:
    prompt = [
        SystemMessage(
            "Eres el asistente de soporte de ACME. Une las respuestas en un solo mensaje para el cliente, en español, "
            "en máximo seis frases. Conserva las citas [archivo, p. N]. No agregues datos nuevos."
        ),
        HumanMessage("\n\n".join(state.get("answers", [])) or f"Sin tareas. Mensaje del cliente: {state['message']}"),
    ]
    return {"reply": model.invoke(prompt).text}


builder = StateGraph(TicketState)
builder.add_node("orchestrator", orchestrator)
builder.add_node("worker", worker)  # el subgrafo, como un nodo más
builder.add_node("synthesize", synthesize)
builder.add_edge(START, "orchestrator")
builder.add_conditional_edges("orchestrator", assign_workers, ["worker", "synthesize"])
builder.add_edge("worker", "synthesize")
builder.add_edge("synthesize", END)
support = builder.compile()
show(support)

In [ ]:
message = (
    "Hola, soy C-3307 y tengo 75 días de mora. Quiero saber cuánto me pueden ampliar el plazo si reestructuro, "
    "si ya me reportaron a centrales de riesgo y si puedo pagar las cuotas por PSE desde otro banco."
)
result = support.invoke({"message": message})

print("Plan:")
for task in result["tasks"]:
    print(f"  · [{task['policy']}] {task['question']}")
print("\nRespuesta al cliente:\n", result["reply"])

Revisa el plan: ¿separó las tres preguntas y eligió la política de cobranza para las dos primeras y la guía de costos para la de PSE? En nuestras pruebas separó bien las preguntas, pero asignó la de reestructuración a la política de **crédito**: el trabajador respondió con la regla de mora para pedir un crédito nuevo, que no aplica, y la síntesis la repitió. **El plan es el punto débil del patrón**: un error del orquestador se propaga a todo lo que sigue. Conviene describir mejor cada política en `Task.policy` (práctica 5) y medir el plan como medimos el router de L14.

## 4. Cuándo usar cada patrón

| | Paralelización (L15) | Orchestrator-workers (L16) |
|---|---|---|
| Número de tareas | Fijo, escrito en el código | Lo decide el modelo en cada ejecución |
| Entrada de cada rama | El estado completo | Su propia entrada, con `Send` |
| Llamadas | Siempre las mismas | Variables: 1 + N + 1 |
| Riesgo principal | Pagar ramas que no hacen falta | Un plan mal dividido |

Usa orchestrator-workers cuando la cantidad o el tipo de subtareas depende de la entrada; si siempre son las mismas, la paralelización fija es más barata y predecible.

## Práctica breve

1. Pasa un ticket con una sola pregunta. ¿Cuántos trabajadores lanza el grafo y cuántas llamadas hace en total?
2. Prueba el subgrafo `worker` solo con `worker.invoke({"question": ..., "policy": ...})`.
3. Cambia `load_policy` para que use el retriever de L4 en lugar de la política completa.
4. Añade un límite: si el plan trae más de 4 tareas, responde pidiendo al cliente que escriba un mensaje por tema.
5. Mejora la descripción de `Task.policy` (por ejemplo, «cobranza: mora, reportes, reestructuración y desempleo») y repite la sección 3. ¿Asigna bien la reestructuración?

## Referencias

- **Anthropic**, [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents): *orchestrator-workers*.
- [`Send` en LangGraph](https://docs.langchain.com/oss/python/langgraph/graph-api#send) · [Subgrafos](https://docs.langchain.com/oss/python/langgraph/use-subgraphs)
- **HuggingGPT** (Shen et al., 2023): un modelo que planea y delega tareas a modelos especialistas. [arXiv:2303.17580](https://arxiv.org/abs/2303.17580)